# RefMet Harmonization Publication Pipeline

This notebook runs the final publication-oriented RefMet harmonization pipeline.

**Goal:** given a lab-provided `metabolite_name`, return the top RefMet candidate names and, when the mapping is uncertain, return PubMed/PubChem synonym/CID evidence for manual curation.

The pipeline is scale-aware:

- The RefMet target vocabulary is relatively small, about 188k names, so it is searched with exact dense retrieval.
- The PubMed/PubChem synonym database is much larger, about 40M names, so it is condensed by CID before embedding. The default keeps the best 20 representative synonym surfaces per CID.
- Large embedding artifacts are cached in scratch during the run and checkpointed back into `results/refmet_harmonization_publication_pipeline/` so a Colab reconnect does not force re-encoding everything.

Main outputs include top-k RefMet predictions, strict non-identical/no-query-surface metrics, reranker weights, bootstrap confidence intervals, confidence routing curves, PubMed/PubChem fallback candidates, and human-review samples.

In [ ]:
# Colab-safe installs. This avoids forcing pandas/numpy upgrades that can conflict with Colab.
import importlib.util
import subprocess
import sys

required = {
    'duckdb': 'duckdb>=1.0.0',
    'pyarrow': 'pyarrow>=15.0.0',
    'transformers': 'transformers>=4.41.0',
    'torch': 'torch',
    'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'sentencepiece': 'sentencepiece',
    'tiktoken': 'tiktoken',
}
missing = [pip_name for import_name, pip_name in required.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

if importlib.util.find_spec('faiss') is None:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])

print('Dependency check complete.')

In [ ]:
from pathlib import Path
import os

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

project_candidates = [
    Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
    Path('/content/drive/MyDrive/ChemSynonymizer'),
    Path.cwd(),
    Path.cwd().parent,
]
PROJECT_DIR = next((p for p in project_candidates if (p / 'data').exists() or (p / 'src').exists()), Path.cwd()).resolve()
os.chdir(PROJECT_DIR)
print('Project directory:', PROJECT_DIR)

## Run Configuration

Edit this cell before running if your paths differ.

The most important runtime/quality controls are:

- `PUBMED_MAX_NAMES_PER_CID`: reduces PubMed/PubChem redundancy. Use 10 for faster, 20 as the default, 30 for more coverage.
- `PUBMED_EMBED_ROW_LIMIT`: limits the condensed fallback search surface for same-day experiments. Set to `None` for the full condensed fallback surface.
- `PUBMED_SEARCH_MODE`: use `stream_gpu` for large fallback surfaces that do not fit as one GPU index.
- `PUBMED_GPU_CHUNK_VECTORS`: corpus vectors per GPU chunk during streaming search. On an 80GB A100, 6M is conservative; 8M-10M may be faster if memory allows.
- `PUBMED_FALLBACK_QUERY_MODE`: leave this as `calibrated` for publication runs. The fallback set is chosen from validation-calibrated confidence thresholds rather than a fixed percentage.
- `AUTO_ACCEPT_TARGET_PRECISION`: target validation precision for high-confidence RefMet auto-accept mappings. Default is 0.90.
- `FALLBACK_TARGET_HIT10`: validation Hit@10 threshold used to decide which low-confidence queries should be sent to PubMed/PubChem fallback. Default is 0.80.
- `PIPELINE_STAGE`: use `gpu` to build embeddings/FAISS artifacts, then `cpu` to do parallel reranker scoring/reporting. Use `all` to run everything in one session.
- `N_JOBS`: CPU process count for reranker scoring in the CPU stage.
- `RERANK_POOL_K`: how many RefMet candidates get lexical/hybrid reranking. Use 128 for speed, 256 for the most exhaustive publication run.
- `RUN_PUBMED_FALLBACK`: set `False` if you only want closed-set RefMet evaluation.

The closed-set RefMet evaluation remains exact; the PubMed/PubChem condensation only affects fallback candidate evidence for possible new RefMet entries.

In [ ]:
import os
from pathlib import Path

RUN_NAME = 'refmet_harmonization_publication_pipeline'
LAB_CSV = PROJECT_DIR / 'data' / 'allmetnames_062626.csv'

# If you have an official RefMet vocabulary export, set this path.
# If left blank, the notebook uses the unique refmet_name values from the lab file.
REFMET_CSV = ''  # Example: str(PROJECT_DIR / 'data' / 'refmet_names_188k.csv')

PUBMED_CSV = PROJECT_DIR / 'data' / 'CID-Synonym-full.csv'
if not PUBMED_CSV.exists():
    PUBMED_CSV = PROJECT_DIR / 'data' / 'good.csv'

MODEL_PATH = PROJECT_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'
MODEL_NAME_OR_PATH = str(MODEL_PATH if MODEL_PATH.exists() else 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext')

# Stage controls.
# Use 'gpu' first to create/cache embeddings and FAISS arrays, then 'cpu' to do parallel scoring/reporting.
# Use 'all' if you want one uninterrupted run.
PIPELINE_STAGE = 'all'  # all, gpu, or cpu
N_JOBS = 8
PREDICTION_CHUNK_SIZE = 4096
FORCE_RESCORE = False

# A100 settings inherited from the earlier tuned SapBERT work.
MAX_LENGTH = 96
POOLING = 'cls'
DEVICE = 'cuda'
REFMET_BATCH_SIZE = 8192
QUERY_BATCH_SIZE = 8192
PUBMED_BATCH_SIZE = 8192
SEARCH_BATCH_SIZE = 8192

# Retrieval and reranking.
TOP_X = 10
K_RETRIEVE_REFMET = 256
RERANK_POOL_K = 128   # Set to 256 for the most exhaustive reranking run.
K_RETRIEVE_PUBMED = 100
EXACT_GPU_MAX_VECTORS = 2_500_000

# PubMed/PubChem fallback scale controls.
RUN_PUBMED_FALLBACK = True
PUBMED_SEARCH_MODE = 'stream_gpu'      # auto, stream_gpu, flat_gpu, or flat_cpu
PUBMED_GPU_CHUNK_VECTORS = 6_000_000  # Try 8_000_000 or 10_000_000 on an 80GB A100 if stable.
PUBMED_FALLBACK_QUERY_MODE = 'calibrated'  # Main publication strategy: validation-calibrated fallback.
AUTO_ACCEPT_TARGET_PRECISION = 0.90
FALLBACK_TARGET_HIT10 = 0.80
PUBMED_MAX_NAMES_PER_CID = 20
PUBMED_EMBED_ROW_LIMIT = 2_000_000  # Set to None for the full condensed fallback index.
REBUILD_PUBMED_CONDENSED = False

# Reranker/uncertainty controls.
RERANK_RANDOM_SEARCH_TRIALS = 240
RERANK_TUNING_MAX_QUERIES = 30_000
BOOTSTRAP_SAMPLES = 1000

env_values = {
    'CHEMSYN_PROJECT_DIR': str(PROJECT_DIR),
    'CHEMSYN_RUN_NAME': RUN_NAME,
    'CHEMSYN_LAB_CSV': str(LAB_CSV),
    'CHEMSYN_PUBMED_CSV': str(PUBMED_CSV),
    'CHEMSYN_MODEL_NAME_OR_PATH': MODEL_NAME_OR_PATH,
    'CHEMSYN_DEVICE': DEVICE,
    'CHEMSYN_PIPELINE_STAGE': PIPELINE_STAGE,
    'CHEMSYN_N_JOBS': str(N_JOBS),
    'CHEMSYN_PREDICTION_CHUNK_SIZE': str(PREDICTION_CHUNK_SIZE),
    'CHEMSYN_FORCE_RESCORE': str(FORCE_RESCORE),
    'CHEMSYN_MAX_LENGTH': str(MAX_LENGTH),
    'CHEMSYN_POOLING': POOLING,
    'CHEMSYN_REFMET_BATCH_SIZE': str(REFMET_BATCH_SIZE),
    'CHEMSYN_QUERY_BATCH_SIZE': str(QUERY_BATCH_SIZE),
    'CHEMSYN_PUBMED_BATCH_SIZE': str(PUBMED_BATCH_SIZE),
    'CHEMSYN_SEARCH_BATCH_SIZE': str(SEARCH_BATCH_SIZE),
    'CHEMSYN_TOP_X': str(TOP_X),
    'CHEMSYN_K_RETRIEVE_REFMET': str(K_RETRIEVE_REFMET),
    'CHEMSYN_RERANK_POOL_K': str(RERANK_POOL_K),
    'CHEMSYN_K_RETRIEVE_PUBMED': str(K_RETRIEVE_PUBMED),
    'CHEMSYN_EXACT_GPU_MAX_VECTORS': str(EXACT_GPU_MAX_VECTORS),
    'CHEMSYN_RUN_PUBMED_FALLBACK': str(RUN_PUBMED_FALLBACK),
    'CHEMSYN_PUBMED_SEARCH_MODE': PUBMED_SEARCH_MODE,
    'CHEMSYN_PUBMED_GPU_CHUNK_VECTORS': str(PUBMED_GPU_CHUNK_VECTORS),
    'CHEMSYN_PUBMED_FALLBACK_QUERY_MODE': PUBMED_FALLBACK_QUERY_MODE,
    'CHEMSYN_AUTO_ACCEPT_TARGET_PRECISION': str(AUTO_ACCEPT_TARGET_PRECISION),
    'CHEMSYN_FALLBACK_TARGET_HIT10': str(FALLBACK_TARGET_HIT10),
    'CHEMSYN_PUBMED_MAX_NAMES_PER_CID': str(PUBMED_MAX_NAMES_PER_CID),
    'CHEMSYN_PUBMED_EMBED_ROW_LIMIT': 'None' if PUBMED_EMBED_ROW_LIMIT is None else str(PUBMED_EMBED_ROW_LIMIT),
    'CHEMSYN_REBUILD_PUBMED_CONDENSED': str(REBUILD_PUBMED_CONDENSED),
    'CHEMSYN_RERANK_RANDOM_SEARCH_TRIALS': str(RERANK_RANDOM_SEARCH_TRIALS),
    'CHEMSYN_RERANK_TUNING_MAX_QUERIES': str(RERANK_TUNING_MAX_QUERIES),
    'CHEMSYN_BOOTSTRAP_SAMPLES': str(BOOTSTRAP_SAMPLES),
}
if REFMET_CSV:
    env_values['CHEMSYN_REFMET_CSV'] = str(REFMET_CSV)
else:
    os.environ.pop('CHEMSYN_REFMET_CSV', None)

for key, value in env_values.items():
    os.environ[key] = value

print('Configured run:', RUN_NAME)
print('Lab CSV:', LAB_CSV)
print('RefMet CSV:', REFMET_CSV or 'using lab refmet_name values')
print('PubMed/PubChem CSV:', PUBMED_CSV)
print('Model:', MODEL_NAME_OR_PATH)
print('PubMed max names per CID:', PUBMED_MAX_NAMES_PER_CID)
print('PubMed fallback row cap:', PUBMED_EMBED_ROW_LIMIT)
print('Pipeline stage:', PIPELINE_STAGE)
print('CPU jobs:', N_JOBS)
print('PubMed search mode:', PUBMED_SEARCH_MODE)
print('PubMed GPU chunk vectors:', PUBMED_GPU_CHUNK_VECTORS)
print('PubMed fallback query mode:', PUBMED_FALLBACK_QUERY_MODE)
print('Auto-accept target precision:', AUTO_ACCEPT_TARGET_PRECISION)
print('Fallback target Hit@10:', FALLBACK_TARGET_HIT10)

In [ ]:
# Run the pipeline. It is safe to rerun: completed artifacts are reused when shapes match.
import subprocess
import sys
from pathlib import Path

script_path = PROJECT_DIR / 'scripts' / 'run_refmet_harmonization_publication_pipeline.py'
if not script_path.exists():
    raise FileNotFoundError(f'Could not find pipeline script: {script_path}')

cmd = [sys.executable, str(script_path)]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
return_code = proc.wait()
if return_code != 0:
    raise RuntimeError(f'Pipeline failed with exit code {return_code}')

In [ ]:
# Inspect saved outputs.
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

RESULTS_DIR = PROJECT_DIR / 'results' / RUN_NAME
print('Results folder:', RESULTS_DIR)

metrics_path = RESULTS_DIR / 'evaluation_metrics.csv'
if metrics_path.exists():
    display(pd.read_csv(metrics_path))

route_path = RESULTS_DIR / 'curation_route_summary.csv'
if route_path.exists():
    display(pd.read_csv(route_path))

tier_path = RESULTS_DIR / 'confidence_tier_diagnostics.csv'
if tier_path.exists():
    display(pd.read_csv(tier_path))

summary_path = RESULTS_DIR / 'publication_pipeline_summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))

print('\nGenerated files:')
for path in sorted(RESULTS_DIR.glob('*')):
    print('-', path.name)

In [ ]:
# Display key plots if they were generated.
from pathlib import Path
from IPython.display import Image, display

plot_dir = RESULTS_DIR / 'plots'
for name in [
    'MRR_comparison.png',
    'Hitat1_comparison.png',
    'Hitat10_comparison.png',
    'confidence_threshold_curve.png',
    'curation_route_distribution.png',
]:
    path = plot_dir / name
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))